# Demo orchestrator

In [1]:
import sys
import os
import random
import torch
import pandas as pd
import logging

# 1. Setup path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. Import Pipeline components
from peptide_pipeline.orchestrator.orchestrator import Orchestrator
from peptide_pipeline.chemist.agent_v1 import ChemistAgent
from peptide_pipeline.chemist.agent_v1.config_chemist import ChemistConfig
from peptide_pipeline.biologist.esm_biologist_zscore import ESMBiologistZscore
from peptide_pipeline.generator.vae_generator import VAEGenerator


/home/niels/pyvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Préparation et Entraînement du Générateur (VAE)

In [2]:
SEQ_LENGTH = 6 
INPUT_DIM = SEQ_LENGTH * 20  # 120
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

# --- 1.1 Génération de données synthétiques ---
print("Generating synthetic training data...")
random.seed(42)
train_peptides = list(set(
    ''.join(random.choices(AMINO_ACIDS, k=SEQ_LENGTH))
    for _ in range(1000)
))

# --- 1.2 Encodage One-Hot ---
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
def encode(sequences, seq_length):
    tensor = torch.zeros(len(sequences), seq_length * 20)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            if j < seq_length:
                tensor[i, j * 20 + aa_index[aa]] = 1.0
    return tensor

train_data = encode(train_peptides, SEQ_LENGTH)

# --- 1.3 Initialisation et Entraînement ---
generator = VAEGenerator(input_dim=INPUT_DIM, latent_dim=32, hidden_dim=128)
print(f"Device: {generator.device}")

print("Starting VAE warm-up training...")
generator.train_model(train_data.to(generator.device), epochs=150, batch_size=64, lr=1e-3)
print("VAE Ready!")

Generating synthetic training data...
Device: cuda
Starting VAE warm-up training...
  Epoch 050/150 | Recon: 0.1532 | KL: 2.2536 | KL weight: 0.05
  Epoch 100/150 | Recon: 0.0624 | KL: 1.6059 | KL weight: 0.10
  Epoch 150/150 | Recon: 0.0593 | KL: 1.4488 | KL weight: 0.15
VAE Ready!


## 2. Initialisation des Agents & Orchestrateur

In [3]:
chemist_config = ChemistConfig() #defaut config ph 7 
chemist = ChemistAgent(config = chemist_config)

# Biologiste : On utilise 'RLLKRF' comme référence (donc on cherche des peptides similaires structurellement)
biologist = ESMBiologistZscore(
    reference_peptide="RLLKRF", 
    model_name="facebook/esm2_t6_8M_UR50D"
)

orchestrator = Orchestrator(generator, chemist, biologist)
print("Orchestrator Initialized.")

Loading weights: 100%|██████████| 107/107 [00:00<00:00, 1445.05it/s]
EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Orchestrator Initialized.


## 3. Exécution de la Pipeline

Nous partons du peptide **"RLLKRF"**. L'orchestrateur va :
1. Demander au VAE de générer des variantes (mutation via l'espace latent).
2. Filtrer chimiquement.
3. Scorer biologiquement.
4. Répéter le processus sur les survivants.

In [4]:
results = orchestrator.run(
    nb_iterations=5,
    nb_peptides=50,
    top_k=5,
    exploration_rate=0.3, # Bruit injecté dans l'espace latent pour créer des variants
    initial_peptide="RLLKRF"
)

print(f"\nPipeline Finished. {len(results)} candidates found.")

Sequence VANTAQ is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence CSEFME is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence KIRHHA is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence CIVGTP is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence TWRHMW is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence NMVRYT is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence NCVFTP is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence KHTQHR is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence RWWMDH is inval


Pipeline Finished. 5 candidates found.


## 4. Résultats

In [5]:
df_results = []
for res in results:
    row = {
        "Peptide": res.get("peptide"),
        "Score ESM": res.get("score"),
        "Score Chem": res.get("chemist_score"),
        "In limits": res.get("in_limits"),
    }
    row.update(res.get("properties", {}))
    df_results.append(row)

df = pd.DataFrame(df_results)
if not df.empty:
    preferred_cols = [
        "Peptide",
        "Score ESM",
        "Score Chem",
        "In limits",
        "molecular_weight",
        "logp",
    ]
    cols = [c for c in preferred_cols if c in df.columns]
    cols += [c for c in df.columns if c not in cols]
    display(df[cols])
else:
    print("Aucun résultat.")

,Peptide,Score ESM,Score Chem,In limits
0,LHRTNF,0.843704,1.0,True
1,IQEWRF,0.840578,1.0,True
2,YLFILA,0.811956,1.0,True
3,KNLRHP,0.809418,1.0,True
4,VIFRDM,0.804202,1.0,True
